# Gold Layer - Business Aggregations

**Purpose:** Create business-ready aggregate tables optimized for analytics, dashboards, and reporting.

**Input Tables:**
* `workspace.default.movies_silver` - Cleaned movie data
* `workspace.default.movies_bronze_genres` - Genre mapping

**Output Tables:**
* `workspace.default.movies_gold_by_genre` - Movie metrics aggregated by genre with human-readable names
* `workspace.default.movies_gold_by_decade` - Movie metrics aggregated by decade for trend analysis

**Aggregations:**
* Count of movies
* Average ratings and popularity
* Total votes and engagement metrics
* Time range (earliest/latest movies)

**Execution Order:** Run this notebook after silver transformations (02-silver-transformations).

In [0]:
%sql
-- Purpose: Create gold layer analytics table aggregated by genre with human-readable genre names
-- Input: workspace.default.movies_silver, workspace.default.movies_bronze_genres
-- Output: workspace.default.movies_gold_by_genre

-- Step 1: Explode genre_ids array for genre-level analysis
CREATE OR REPLACE TEMP VIEW movies_with_genres AS
SELECT
  movie_id,
  title,
  release_year,
  release_decade,
  vote_average,
  vote_count,
  popularity,
  CAST(genre_id AS INT) AS genre_id
FROM
  workspace.default.movies_silver
LATERAL VIEW explode(from_json(genre_ids, 'array<int>')) AS genre_id;

-- Step 2: Aggregate metrics by genre
CREATE OR REPLACE TEMP VIEW genre_aggregates AS
SELECT
  genre_id,
  COUNT(DISTINCT movie_id) AS total_movies,
  ROUND(AVG(vote_average), 2) AS avg_rating,
  ROUND(AVG(popularity), 2) AS avg_popularity,
  SUM(vote_count) AS total_votes,
  MIN(release_year) AS earliest_movie_year,
  MAX(release_year) AS latest_movie_year
FROM
  movies_with_genres
GROUP BY
  genre_id;

-- Step 3: Join with genre names and create gold table
CREATE OR REPLACE TABLE workspace.default.movies_gold_by_genre
COMMENT 'Gold layer: Movie metrics aggregated by genre with human-readable genre names. Updated on-demand after silver refresh. Source: workspace.default.movies_silver joined with workspace.default.movies_bronze_genres.'
AS
SELECT
  g.genre_name,
  ga.total_movies,
  ga.avg_rating,
  ga.avg_popularity,
  ga.total_votes,
  ga.earliest_movie_year,
  ga.latest_movie_year
FROM
  genre_aggregates ga
INNER JOIN
  workspace.default.movies_bronze_genres g
  ON ga.genre_id = g.genre_id
ORDER BY
  ga.total_movies DESC;

-- Add column comments for business metrics
ALTER TABLE workspace.default.movies_gold_by_genre
ALTER COLUMN genre_name
COMMENT 'Human-readable genre name';

ALTER TABLE workspace.default.movies_gold_by_genre
ALTER COLUMN avg_rating
COMMENT 'Average TMDB user rating (0-10 scale) across all movies in this genre';

ALTER TABLE workspace.default.movies_gold_by_genre
ALTER COLUMN avg_popularity
COMMENT 'Average TMDB popularity score for movies in this genre';

ALTER TABLE workspace.default.movies_gold_by_genre
ALTER COLUMN total_votes
COMMENT 'Sum of all user votes across movies in this genre';

-- Display results
SELECT
  *
FROM
  workspace.default.movies_gold_by_genre;

In [0]:
%sql
-- Purpose: Create gold layer table with movie metrics aggregated by decade
-- Input: workspace.default.movies_silver
-- Output: workspace.default.movies_gold_by_decade

-- Step 1: Aggregate metrics by decade
CREATE OR REPLACE TEMP VIEW decade_aggregates AS
SELECT
  release_decade,
  COUNT(DISTINCT movie_id) AS total_movies,
  ROUND(AVG(vote_average), 2) AS avg_rating,
  ROUND(AVG(popularity), 2) AS avg_popularity,
  SUM(vote_count) AS total_votes,
  ROUND(AVG(vote_count), 0) AS avg_votes_per_movie
FROM
  workspace.default.movies_silver
WHERE
  release_decade IS NOT NULL
GROUP BY
  release_decade;

-- Step 2: Create gold table for decade analysis
CREATE OR REPLACE TABLE workspace.default.movies_gold_by_decade
COMMENT 'Gold layer: Movie metrics aggregated by decade for trend analysis. Updated on-demand after silver refresh. Source: workspace.default.movies_silver.'
AS
SELECT
  *
FROM
  decade_aggregates
ORDER BY
  release_decade;

-- Add column comments for business metrics
ALTER TABLE workspace.default.movies_gold_by_decade
ALTER COLUMN avg_rating
COMMENT 'Average TMDB user rating (0-10 scale) for movies released in this decade';

ALTER TABLE workspace.default.movies_gold_by_decade
ALTER COLUMN avg_popularity
COMMENT 'Average TMDB popularity score for movies in this decade';

ALTER TABLE workspace.default.movies_gold_by_decade
ALTER COLUMN avg_votes_per_movie
COMMENT 'Average number of user votes per movie in this decade';

-- Display results
SELECT
  *
FROM
  workspace.default.movies_gold_by_decade;